In [1]:
from meles.dataset import read_ytfaces, truncate_dataset, split_dataset
%load_ext autoreload
%autoreload 2

from meles.recognizer.arcface import ArcFaceRecognizer
from meles.aggregation.aggregator import NoAggregator, Aggregator
from meles.aggregation.clustering import MeanClusteringAggregator, MedianClusteringAggregator
from meles.aggregation.pooling import MeanPoolingAggregator, MaxPoolingAggregator, MinPoolingAggregator, \
    MedianPoolingAggregator
from meles.classifier import Classifier
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Allow the usage of progress bars on pandas dataframes
tqdm.pandas()

In [2]:
DATA_PATH = Path("../../data")

dataset = read_ytfaces(DATA_PATH / "ytfaces")
dataset = truncate_dataset(
    dataset,
    max_frames_per_video=300,
    min_videos_per_identity=3
)
dataset

,identity,video,frame,frame_suffix,path
0,Aaron_Eckhart,0,aligned_detect_0.555.jpg,.jpg,Aaron_Eckhart/0/aligned_detect_0.555.jpg
1,Aaron_Eckhart,0,aligned_detect_0.556.jpg,.jpg,Aaron_Eckhart/0/aligned_detect_0.556.jpg
2,Aaron_Eckhart,0,aligned_detect_0.557.jpg,.jpg,Aaron_Eckhart/0/aligned_detect_0.557.jpg
3,Aaron_Eckhart,0,aligned_detect_0.558.jpg,.jpg,Aaron_Eckhart/0/aligned_detect_0.558.jpg
4,Aaron_Eckhart,0,aligned_detect_0.559.jpg,.jpg,Aaron_Eckhart/0/aligned_detect_0.559.jpg
...,...,...,...,...,...
247788,Zulfiqar_Ahmed,4,aligned_detect_4.263.jpg,.jpg,Zulfiqar_Ahmed/4/aligned_detect_4.263.jpg
247789,Zulfiqar_Ahmed,4,aligned_detect_4.264.jpg,.jpg,Zulfiqar_Ahmed/4/aligned_detect_4.264.jpg
247790,Zulfiqar_Ahmed,4,aligned_detect_4.265.jpg,.jpg,Zulfiqar_Ahmed/4/aligned_detect_4.265.jpg
247791,Zulfiqar_Ahmed,4,aligned_detect_4.266.jpg,.jpg,Zulfiqar_Ahmed/4/aligned_detect_4.266.jpg


In [32]:
train, test = split_dataset(dataset, train_videos=2)

In [18]:
recognizer = ArcFaceRecognizer()
aggregators = [
    MedianClusteringAggregator(recognizer, distance_threshold=None, min_size=3, n_clusters=3),
]

In [ ]:
for aggregator in aggregators:


In [3]:
dataset.groupby(["identity", "video"]).apply(lambda x: print(x))

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



                           frame frame_suffix  \
222349  aligned_detect_5.100.jpg         .jpg   
222350  aligned_detect_5.101.jpg         .jpg   
222351  aligned_detect_5.102.jpg         .jpg   
222352  aligned_detect_5.103.jpg         .jpg   
222353  aligned_detect_5.104.jpg         .jpg   
...                          ...          ...   
222585   aligned_detect_5.95.jpg         .jpg   
222586   aligned_detect_5.96.jpg         .jpg   
222587   aligned_detect_5.97.jpg         .jpg   
222588   aligned_detect_5.98.jpg         .jpg   
222589   aligned_detect_5.99.jpg         .jpg   

                                             path  
222349  Sara_Silverman/5/aligned_detect_5.100.jpg  
222350  Sara_Silverman/5/aligned_detect_5.101.jpg  
222351  Sara_Silverman/5/aligned_detect_5.102.jpg  
222352  Sara_Silverman/5/aligned_detect_5.103.jpg  
222353  Sara_Silverman/5/aligned_detect_5.104.jpg  
...                                           ...  
222585   Sara_Silverman/5/aligned_detect_5.95.j

,frame,frame_suffix,path


In [4]:
import pandas as pd

def k_fold_splits(
        df: pd.DataFrame,
        k: int,
        identity_col: str = "identity",
        video_col: str = "video",
        shuffle: bool = True,
        random_state: int | None = 0,
) -> list[pd.DataFrame]:
    """k copies of df, each with a bool `train` column. Splits happen per-identity
    at video granularity: every video stays whole, every identity appears in
    both train and test of every fold."""

    # 1. one row per (identity, video)
    videos = df[[identity_col, video_col]].drop_duplicates()

    # 2. guard the edge case: an identity with < k videos can't reach every fold
    counts = videos.groupby(identity_col).size()
    if (counts < k).any():
        bad = counts[counts < k].to_dict()
        raise ValueError(f"identities with fewer than k={k} videos: {bad}")

    # 3. assign each video a fold index within its identity (round-robin)
    if shuffle:
        videos = videos.sample(frac=1, random_state=random_state)
    videos["_fold"] = videos.groupby(identity_col).cumcount() % k

    # 4. propagate the fold index to every frame
    df = df.merge(videos, on=[identity_col, video_col], how="left")

    # 5. fold i is the test fold; everything else is train
    return [
        df.drop(columns="_fold").assign(train=(df["_fold"] != i))
        for i in range(k)
    ]

In [7]:
[k1, k2, k3] = k_fold_splits(dataset, 3)

In [8]:
k1

,identity,video,frame,frame_suffix,path,train
0,Aaron_Eckhart,0,aligned_detect_0.555.jpg,.jpg,Aaron_Eckhart/0/aligned_detect_0.555.jpg,True
1,Aaron_Eckhart,0,aligned_detect_0.556.jpg,.jpg,Aaron_Eckhart/0/aligned_detect_0.556.jpg,True
2,Aaron_Eckhart,0,aligned_detect_0.557.jpg,.jpg,Aaron_Eckhart/0/aligned_detect_0.557.jpg,True
3,Aaron_Eckhart,0,aligned_detect_0.558.jpg,.jpg,Aaron_Eckhart/0/aligned_detect_0.558.jpg,True
4,Aaron_Eckhart,0,aligned_detect_0.559.jpg,.jpg,Aaron_Eckhart/0/aligned_detect_0.559.jpg,True
...,...,...,...,...,...,...
247788,Zulfiqar_Ahmed,4,aligned_detect_4.263.jpg,.jpg,Zulfiqar_Ahmed/4/aligned_detect_4.263.jpg,True
247789,Zulfiqar_Ahmed,4,aligned_detect_4.264.jpg,.jpg,Zulfiqar_Ahmed/4/aligned_detect_4.264.jpg,True
247790,Zulfiqar_Ahmed,4,aligned_detect_4.265.jpg,.jpg,Zulfiqar_Ahmed/4/aligned_detect_4.265.jpg,True
247791,Zulfiqar_Ahmed,4,aligned_detect_4.266.jpg,.jpg,Zulfiqar_Ahmed/4/aligned_detect_4.266.jpg,True


In [9]:
k2

,identity,video,frame,frame_suffix,path,train
0,Aaron_Eckhart,0,aligned_detect_0.555.jpg,.jpg,Aaron_Eckhart/0/aligned_detect_0.555.jpg,True
1,Aaron_Eckhart,0,aligned_detect_0.556.jpg,.jpg,Aaron_Eckhart/0/aligned_detect_0.556.jpg,True
2,Aaron_Eckhart,0,aligned_detect_0.557.jpg,.jpg,Aaron_Eckhart/0/aligned_detect_0.557.jpg,True
3,Aaron_Eckhart,0,aligned_detect_0.558.jpg,.jpg,Aaron_Eckhart/0/aligned_detect_0.558.jpg,True
4,Aaron_Eckhart,0,aligned_detect_0.559.jpg,.jpg,Aaron_Eckhart/0/aligned_detect_0.559.jpg,True
...,...,...,...,...,...,...
247788,Zulfiqar_Ahmed,4,aligned_detect_4.263.jpg,.jpg,Zulfiqar_Ahmed/4/aligned_detect_4.263.jpg,False
247789,Zulfiqar_Ahmed,4,aligned_detect_4.264.jpg,.jpg,Zulfiqar_Ahmed/4/aligned_detect_4.264.jpg,False
247790,Zulfiqar_Ahmed,4,aligned_detect_4.265.jpg,.jpg,Zulfiqar_Ahmed/4/aligned_detect_4.265.jpg,False
247791,Zulfiqar_Ahmed,4,aligned_detect_4.266.jpg,.jpg,Zulfiqar_Ahmed/4/aligned_detect_4.266.jpg,False


In [10]:
k3

,identity,video,frame,frame_suffix,path,train
0,Aaron_Eckhart,0,aligned_detect_0.555.jpg,.jpg,Aaron_Eckhart/0/aligned_detect_0.555.jpg,False
1,Aaron_Eckhart,0,aligned_detect_0.556.jpg,.jpg,Aaron_Eckhart/0/aligned_detect_0.556.jpg,False
2,Aaron_Eckhart,0,aligned_detect_0.557.jpg,.jpg,Aaron_Eckhart/0/aligned_detect_0.557.jpg,False
3,Aaron_Eckhart,0,aligned_detect_0.558.jpg,.jpg,Aaron_Eckhart/0/aligned_detect_0.558.jpg,False
4,Aaron_Eckhart,0,aligned_detect_0.559.jpg,.jpg,Aaron_Eckhart/0/aligned_detect_0.559.jpg,False
...,...,...,...,...,...,...
247788,Zulfiqar_Ahmed,4,aligned_detect_4.263.jpg,.jpg,Zulfiqar_Ahmed/4/aligned_detect_4.263.jpg,True
247789,Zulfiqar_Ahmed,4,aligned_detect_4.264.jpg,.jpg,Zulfiqar_Ahmed/4/aligned_detect_4.264.jpg,True
247790,Zulfiqar_Ahmed,4,aligned_detect_4.265.jpg,.jpg,Zulfiqar_Ahmed/4/aligned_detect_4.265.jpg,True
247791,Zulfiqar_Ahmed,4,aligned_detect_4.266.jpg,.jpg,Zulfiqar_Ahmed/4/aligned_detect_4.266.jpg,True
